#### Worker

In [1]:
from app.db import get_db, db_close,insert_scan_queue,update_scan_item_status, pull_next_scan, SCAN_QUEUE, QueueItemStatus
from motor.motor_asyncio import AsyncIOMotorDatabase
from datetime import datetime

In [2]:
import asyncio

async def scan(woker_id: int, url: str):
    await asyncio.sleep(2)

In [3]:
async def worker(worker_id: int, db: AsyncIOMotorDatabase):
    pull_frequency = 1.0 

    while True:
         # check if new item in queue with created status
        item_to_scan = await pull_next_scan(db)

        if not item_to_scan:
            await asyncio.sleep(pull_frequency)

        else:   
            url = item_to_scan['url_name']
            print(f"WORKER {worker_id}: Scanning:{url}")

            try: 
                await scan(woker_id = worker_id, url=url)
                await update_scan_item_status(db = db, id = item_to_scan['id'], status=QueueItemStatus.COMPLETED, woker_id = worker_id)
                
            except Exception as e:
                print(f"Scan failed with ecpetion:{e}")
    

In [4]:
async def seed_scan_queue(db: AsyncIOMotorDatabase) -> None:
    docs = [
        {                                                                                                     
            "id": i,
            "url_name": f"http://localhost:8080/page_{i}.php",                                                
            "status": QueueItemStatus.CREATED,                                                                
            "created_at": datetime.now(),
            "scanned_by": None,                                                                               
            "scanned_at": None,
        }                                                                                                     
          for i in range(1, 6)
      ]
    await db[SCAN_QUEUE].insert_many(docs)  

In [5]:
async def main():
    db_ = await get_db("mongodb://localhost:27017", "crawl-task")
    await seed_scan_queue(db=db_)

    await asyncio.gather(
        worker(1, db_)
    )
    await db_close()

In [ ]:
await main()

Mongo DB connected
WORKER 1: Scanning:http://localhost:8080/page_1.php
Scan failed with ecpetion:update_scan_item_status() got an unexpected keyword argument 'id'
WORKER 1: Scanning:http://localhost:8080/page_2.php
Scan failed with ecpetion:update_scan_item_status() got an unexpected keyword argument 'id'
WORKER 1: Scanning:http://localhost:8080/page_3.php
Scan failed with ecpetion:update_scan_item_status() got an unexpected keyword argument 'id'
WORKER 1: Scanning:http://localhost:8080/page_4.php
